In [14]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re

import ipywidgets as widgets
from IPython.display import display, clear_output

#### methods to load data

In [15]:
# Load data methods
def load_one_trial(trial_file):
    """
    Loads a single trial CSV file into a pandas DataFrame.
    
    Parameters:
    trial_file (str): Path to the target CSV file.
    
    Returns:
    DataFrame: The loaded trial data.
    """
    return pd.read_csv(trial_file)

def load_all_trials(participant_dir):
    """
    Loads and combines all individual trial CSV files for a single participant.
    
    Parameters:
    participant_dir (str): Path to the participant's folder containing CSVs.
    
    Returns:
    DataFrame: A single, continuous DataFrame containing all trials.
    """
    all_dfs = []

    # Loop through all files in the folder
    for file_name in os.listdir(participant_dir):
        # Only process files that end with .csv
        if file_name.endswith('.csv'):
            # Construct the full path to the file
            file_path = os.path.join(participant_dir, file_name)

            # Load the individual trial and add it to our list
            trial_df = load_one_trial(file_path)
            all_dfs.append(trial_df)

    # Combine all individual DataFrames into a single continuous DataFrame
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        return pd.DataFrame()  # Return empty DataFrame if no CSV files found
def load_all_participants(data_dir,append=True):
    """
    Iterates through all participant directories within a root data directory.
    
    Parameters:
    data_dir (str): Root directory containing all participant folders.
    append (bool): If True, merges everything into one DataFrame. 
                   If False, returns a list of separate DataFrames.
                   
    Returns:
    DataFrame or list: Combined dataset or a list of participant DataFrames.
    """
    all_dfs = []
    # Loop through all files in the folder
    for participant_dir in os.listdir(data_dir):
            # Construct the full path to the file
            file_path = os.path.join(data_dir, participant_dir)
            participant_df = load_all_trials(file_path)
            participant_df["participant_id"] = participant_dir
            all_dfs.append(participant_df)
    if append:
        return pd.concat(all_dfs,ignore_index=True)
    else:
        return all_dfs



#### Load the data

In [16]:
# Add the block number manually to the csv for data before 16.06.26 where this isn't the case already
# folder = Path(data_dir +"/UB71")

# for file in folder.glob("*.csv"):
#     # Extract block number (allows negative numbers)
#     match = re.search(r'block_(-?\d+)', file.stem)

#     if match:
#         block = int(match.group(1))

#         df = pd.read_csv(file)
#         df["block"] = block
#         df.to_csv(file, index=False)

#         print(f"{file.name}: block = {block}")

In [17]:
data_dir = "C:/Users/HP/Documents/uni/SoSe26/expra/code/analysis/data"
data = load_all_participants(data_dir)
data


  
# # correct corrupted data (only do this for ge03 testdata with the bug not fixed)
# cursor_x_cm = data["gaze_x"]*(59.7/33) 
# cursor_y_cm = data["gaze_y"]
# gaze_x = data["cursor_x_cm"]
# gaze_y = data["cursor_y_cm"]
# data["cursor_x_cm"] = cursor_x_cm
# data["cursor_y_cm"] = cursor_y_cm
# data["gaze_x"] = gaze_x
# data["gaze_y"] = gaze_y
# data["block"] = 0
# data

,trial,time,cursor_x_pix,cursor_y_pix,cursor_x_cm,cursor_y_cm,gaze_x,gaze_y,target_x,target_y,state_marker,button_pressed,start_time,block,participant_id
0,0,9.121739,-1027.0,-718.0,-23.949961,-16.803194,100.900024,-129.500000,-30.085829,239.245228,0,False,1.230487,-1,UB71
1,0,9.151541,-1027.0,-718.0,-23.949961,-16.803194,94.300049,-124.700012,-30.085829,239.245228,0,False,1.230487,-1,UB71
2,0,9.158054,-1027.0,-718.0,-23.949961,-16.803194,94.900024,-121.500000,-30.085829,239.245228,0,False,1.230487,-1,UB71
3,0,9.164816,-1027.0,-718.0,-23.949961,-16.803194,95.699951,-117.900024,-30.085829,239.245228,0,False,1.230487,-1,UB71
4,0,9.171733,-1027.0,-718.0,-23.949961,-16.803194,93.800049,-118.000000,-30.085829,239.245228,0,False,1.230487,-1,UB71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55476,9,697.853616,275.0,151.0,6.413086,3.533819,325.900024,139.700012,287.001231,166.906342,4,True,1.301589,3,UB71
55477,9,697.860703,275.0,151.0,6.413086,3.533819,317.300049,133.799988,287.001231,166.906342,4,True,1.301589,3,UB71
55478,9,697.867733,275.0,151.0,6.413086,3.533819,324.699951,135.799988,287.001231,166.906342,4,True,1.301589,3,UB71
55479,9,697.874278,275.0,151.0,6.413086,3.533819,325.400024,137.000000,287.001231,166.906342,4,True,1.301589,3,UB71


### methods to visualize trials

In [26]:
def visualize_trial(data,participant_id="ge03",block_n=0,trial_n=0):
    # 1. Load the trial data
    df = data[data["participant_id"] == participant_id]
    df = data[data["block"] == block_n]
    df = df[df["trial"] == trial_n]


    # 2. Extract unique trial number and target positions for the plot title/markers
    # (Assuming target coordinates and trial ID remain constant within a single trial file)
    target_x = df["target_x"].iloc[0]
    target_y = df["target_y"].iloc[0]

    # 3. Create the plot
    plt.figure(figsize=(10, 8))

    # wrong screen width and pixel amount was used in params leading to conversion factor
    # and cursor_cm is stored in gaze (was switched in the update function by mistake)

    # Plot Cursor Trajectory in cm
    plt.plot(
        df["cursor_x_pix"],
        df["cursor_y_pix"],
        label="Cursor Path",
        color="royalblue",
        linewidth=2,
        marker="o",
        markersize=3,
        alpha=0.7,
    )


    # gaze is actually stored in the cursor variable (was switched in the update function by mistake)

    # Plot Gaze Trajectory
    plt.plot(
        df["gaze_x"],
        df["gaze_y"], 
        label="Gaze Path",
        color="crimson",
        linewidth=1.5,
        linestyle="--",
        alpha=0.5,
    )

    # Highlight Start Point of the cursor
    plt.scatter(
        0,
        -360,
        color="green",
        s=100,
        zorder=5,
        label="Start Position",
    )

    # # Highlight End Point of the cursor
    # plt.scatter(
    #     df["cursor_x_cm"].iloc[-1],
    #     df["cursor_y_cm"].iloc[-1],
    #     color="red",
    #     s=100,
    #     zorder=5,
    #     label="End Position",
    # )

    # Plot Target Location
    plt.scatter(
        target_x,
        target_y,
        color="gold",
        edgecolors="black",
        s=200,
        marker="*",
        zorder=6,
        label="Target",
    )

    # 4. Customize plot appearance
    plt.title(f"Participant: {participant_id} - Block: {block_n} - Trial: {trial_n}", fontsize=14)
    plt.xlabel("X Position (pix)", fontsize=12)
    plt.ylabel("Y Position (pix)", fontsize=12)
    plt.legend(loc="upper left")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.axis("equal")  # Keeps the aspect ratio 1:1 so distances aren't distorted

    # 5. Display the plot
    plt.show()



### Visualize trial GUI

In [ ]:

# 2. Set up the state variables
participant_list = data['participant_id'].unique().tolist()
total_participants = len(participant_list)

# Track indices instead of hardcoded totals
current_participant = 0
current_block = 0
current_trial = 0

# 3. Create the UI components
output_area = widgets.Output()
prev_participant_button = widgets.Button(description="Prev participant", icon="arrow-left")
next_participant_button = widgets.Button(description="Next participant", icon="arrow-right")
prev_block_button = widgets.Button(description="Prev Block", icon="arrow-left")
next_block_button = widgets.Button(description="Next Block", icon="arrow-right")
prev_trial_button = widgets.Button(description="Prev Trial", icon="arrow-left")
next_trial_button = widgets.Button(description="Next Trial", icon="arrow-right")

# 4. Define button click behaviors

def on_prev_participant_clicked(b):
    global current_participant, current_block, current_trial
    if current_participant > 0:
        current_participant -= 1
        current_block = 0  # Reset to first block of new participant
        current_trial = 0  # Reset to first trial of new block
        update_view()

def on_next_participant_clicked(b):
    global current_participant, current_block, current_trial
    if current_participant < total_participants - 1:
        current_participant += 1
        current_block = 0  # Reset to first block of new participant
        current_trial = 0  # Reset to first trial of new block
        update_view()

def on_prev_block_clicked(b):
    global current_block, current_trial
    if current_block > 0:
        current_block -= 1
        current_trial = 0  # Reset to first trial of new block
        update_view()

def on_next_block_clicked(b):
    global current_block, current_trial
    # Bounds check uses dynamic data
    participant_id = participant_list[current_participant]
    total_blocks = data[data['participant_id'] == participant_id]['block'].nunique()
    
    if current_block < total_blocks - 1:
        current_block += 1
        current_trial = 0  # Reset to first trial of new block
        update_view()

def on_prev_trial_clicked(b):
    global current_trial
    if current_trial > 0:
        current_trial -= 1
        update_view()

def on_next_trial_clicked(b):
    global current_trial
    # Bounds check uses dynamic data
    participant_id = participant_list[current_participant]
    participant_data = data[data['participant_id'] == participant_id]
    block_list = participant_data['block'].unique().tolist()
    block_n = block_list[current_block]
    total_trials = participant_data[participant_data['block'] == block_n]['trial'].nunique()
    
    if current_trial < total_trials - 1:
        current_trial += 1
        update_view()

# 5. Define the update logic
def update_view():
    global current_block, current_trial
    
    # Extract current active IDs
    participant_id = participant_list[current_participant]
    participant_data = data[data['participant_id'] == participant_id]
    
    # Dynamically find available blocks
    block_list = sorted(participant_data['block'].unique().tolist())
    total_blocks = len(block_list)
    
    # Fallback safety check if indices get out of bounds during switch
    if current_block >= total_blocks:
        current_block = total_blocks - 1
        
    block_n = block_list[current_block]
    
    # Dynamically find available trials for this block
    # Note: Assumes your column name is 'trial_n'. Change if it's 'trial_id'.
    trial_list = sorted(participant_data[participant_data['block'] == block_n]['trial'].unique().tolist())
    total_trials = len(trial_list)
    
    if current_trial >= total_trials:
        current_trial = total_trials - 1
        
    trial_n = trial_list[current_trial]

    # Render the visualization
    with output_area:
        clear_output(wait=True)
        
        visualize_trial(data=data, participant_id=participant_id,block_n=block_n, trial_n=trial_n)
    
    # Disable buttons if at boundaries
    prev_participant_button.disabled = (current_participant == 0)
    next_participant_button.disabled = (current_participant == total_participants - 1)
    prev_block_button.disabled = (current_block == 0)
    next_block_button.disabled = (current_block == total_blocks - 1)
    prev_trial_button.disabled = (current_trial == 0)
    next_trial_button.disabled = (current_trial == total_trials - 1)

# 6. Bind callbacks and display the dashboard
prev_participant_button.on_click(on_prev_participant_clicked)
next_participant_button.on_click(on_next_participant_clicked)
prev_block_button.on_click(on_prev_block_clicked)
next_block_button.on_click(on_next_block_clicked)
prev_trial_button.on_click(on_prev_trial_clicked)
next_trial_button.on_click(on_next_trial_clicked)

button_layout = widgets.HBox([prev_participant_button, next_participant_button, prev_block_button, next_block_button, prev_trial_button, next_trial_button])
display(button_layout, output_area)

# Trigger the initial render
update_view()


Output()

In [24]:
data_block_m1 = data[data['block'] == -1]
data_block_0 = data[data['block'] == 0]
data_block_1 = data[data['block'] == 1]
data_block_2 = data[data['block'] == 2]
data_block_3 = data[data['block'] == 3]
print(sorted(data_block_2['trial'].unique()))

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29)]
